In [2]:
all_dataset_names = [
    "AbdomenUS",
    "Acdc",
    "BAGLS",
    "Bbbc010",
    "BkaiIgh",
    "BriFiSeg",
    "Btcv",
    "Busi",
    "CellNuclei",
    "Chaos",
    "ChaseDB1",
    "Chuac",
    "Covid19Radio",
    "CovidQUEx",
    "CystoFluid",
    "Dca1",
    "Deepbacs",
    "Drive",
    "DynamicNuclear",
    "EMSegmentation",
    "FHPsAOP",
    "Idrib",
    "Isic2016",
    "Isic2018",
    "Kvasir",
    "M2caiSeg",
    "MmWhsMr",
    "Monusac",
    "MosMedPlus",
    "Nuclei",
    "Nuset",
    "Pandental",
    "PolypGen",
    "Promise12",
    "RoboTool",
    "TnbcNuclei",
    "UltrasoundNerve",
    "USforKidney",
    "UwSkinCancer",
    "Wbc",
    "Yeaz",
]

def find_latest_runs_per_dataset_filtered_by_conditions(experiments_csv_path, conditions):
    import pandas as pd

    all_experiments = pd.read_csv(experiments_csv_path)

    matching_experiments = all_experiments
    for column_name, column_value in conditions.items():
        matching_experiments = matching_experiments[matching_experiments[column_name] == column_value]

    condition_description = ", ".join(f"{column_name} == {column_value}" for column_name, column_value in conditions.items())

    print(f"Conditions: {condition_description}")
    print(f"Total matching entries in csv: {len(matching_experiments)}")

    if matching_experiments.empty:
        return matching_experiments

    matching_experiments = matching_experiments.sort_values("date")
    latest_run_per_dataset = matching_experiments.groupby("dataset").tail(1)
    latest_run_per_dataset = latest_run_per_dataset.sort_values("dataset")

    # display(latest_run_per_dataset)

    return latest_run_per_dataset


def check_missing_datasets_for_experiment(filtered_experiments, all_dataset_names):
    datasets_present = set(filtered_experiments["dataset"].unique())
    datasets_missing = sorted(set(all_dataset_names) - datasets_present)

    print(f"Total datasets: {len(all_dataset_names)}")
    print(f"Datasets present: {len(datasets_present)}")
    print(f"Datasets missing: {len(datasets_missing)}")
    if datasets_missing:
        print(f"Missing datasets: {datasets_missing}")

    return datasets_missing

In [15]:
latest_runs = find_latest_runs_per_dataset_filtered_by_conditions(
    "experiments/experiments.csv",
    {
        "hypothesis": "Depth-wise separable layers (8-16-32-52) does not lose much in Dice.",
        # "date": "2026-07-17",
        # "architecture_use_depthwise_separable_convolutions": True,
        # "training_batch_size": 32,
        # "architecture_bottleneck_channels": 52
    }
)

check_missing_datasets_for_experiment(latest_runs, all_dataset_names)

Conditions: hypothesis == Depth-wise separable layers (8-16-32-52) does not lose much in Dice.
Total matching entries in csv: 40
Total datasets: 41
Datasets present: 40
Datasets missing: 1
Missing datasets: ['BAGLS']


['BAGLS']

In [12]:
import pandas as pd

latest_runs = find_latest_runs_per_dataset_filtered_by_conditions(
    "experiments/experiments_large.csv",
    {
        "date": "2026-07-17",
        "architecture_use_depthwise_separable_convolutions": True,
        "training_batch_size": 32,
        "architecture_bottleneck_channels": 52
    }
)

# Extract run_ids from the DataFrame
if isinstance(latest_runs, pd.DataFrame):
    latest_runs = latest_runs['run_id'].tolist()

print(f"Matched run_ids: {latest_runs}")
print(f"Total matched: {len(latest_runs)}\n")

paths = ["experiments/experiments_large.csv", "experiments/experiments.csv"]

for path in paths:
    table = pd.read_csv(path, engine="python", on_bad_lines="warn")
    table.to_csv(path, index=False)

new_hypothesis = "Depth-wise separable layers (8-16-32-52) does not lose much in Dice."
new_notes = "Depth-wise separable layers (8-16-32-52)"

for path in paths:
    table = pd.read_csv(path)
    rows_to_update = table["run_id"].isin(latest_runs)
    table.loc[rows_to_update, "hypothesis"] = new_hypothesis
    table.loc[rows_to_update, "notes"] = new_notes
    table.to_csv(path, index=False)
    print(f"{path}: updated {int(rows_to_update.sum())} rows")

Conditions: date == 2026-07-17, architecture_use_depthwise_separable_convolutions == True, training_batch_size == 32, architecture_bottleneck_channels == 52
Total matching entries in csv: 40
Matched run_ids: ['2026_07_17_03_12_14_1750147', '2026_07_17_03_09_43_1750143', '2026_07_17_03_13_15_1750148', '2026_07_17_03_13_44_1750149', '2026_07_17_03_14_45_1750150', '2026_07_17_03_10_13_1750144', '2026_07_17_03_15_19_1750151', '2026_07_17_03_15_45_1750152', '2026_07_17_03_11_13_1750145', '2026_07_17_03_16_15_1750153', '2026_07_17_03_16_51_1750154', '2026_07_17_03_17_46_1750155', '2026_07_17_03_18_16_1750156', '2026_07_17_03_18_47_1750157', '2026_07_17_03_20_16_1750158', '2026_07_17_03_20_50_1750159', '2026_07_17_03_23_20_1750160', '2026_07_17_03_23_47_1750161', '2026_07_17_03_24_17_1750162', '2026_07_17_03_24_17_1750163', '2026_07_17_03_25_51_1750164', '2026_07_17_03_26_20_1750165', '2026_07_17_03_26_21_1750166', '2026_07_17_03_27_23_1750167', '2026_07_17_03_29_17_1750168', '2026_07_17_03_1

In [5]:
import pandas as pd
from pathlib import Path


EXPERIMENTS_LARGE_CSV_PATH = Path("experiments/experiments_large.csv")


def compare_last_two_runs_per_dataset_for_hypothesis(hypothesis_text):
    all_experiments = pd.read_csv(EXPERIMENTS_LARGE_CSV_PATH)

    matching_experiments = all_experiments[
        all_experiments["dataset"].isin(all_dataset_names) &
        (all_experiments["hypothesis"].str.strip() == hypothesis_text)
    ].copy()

    if matching_experiments.empty:
        print(f"No experiments found for hypothesis: {hypothesis_text!r}")
        return

    matching_experiments = matching_experiments.sort_values("run_id")
    last_two_runs_per_dataset = matching_experiments.groupby("dataset").tail(2)

    print(f"Hypothesis: {hypothesis_text!r}\n")
    print(f"{'Dataset':<25} {'Run ID':<25} {'Mean Val Dice':<16} {'Std Val Dice'}")
    print("-" * 85)

    for dataset_name in all_dataset_names:
        dataset_runs = last_two_runs_per_dataset[
            last_two_runs_per_dataset["dataset"] == dataset_name
        ].sort_values("run_id")

        if dataset_runs.empty:
            print(f"{dataset_name:<25} {'(no runs found)'}")
            print()
            continue

        for _, run_row in dataset_runs.iterrows():
            print(
                f"{run_row['dataset']:<25} {run_row['run_id']:<25} "
                f"{run_row['mean_val_dice']:<16.4f} {run_row['std_val_dice']:.4f}"
            )
        if len(dataset_runs) == 2:
            dice_change = dataset_runs.iloc[1]["mean_val_dice"] - dataset_runs.iloc[0]["mean_val_dice"]
            print(f"{'':25} {'Δ dice':<25} {dice_change:+.4f}")
        print()


compare_last_two_runs_per_dataset_for_hypothesis("No hypothesis -  data augmentation + instance norm.")

Hypothesis: 'No hypothesis -  data augmentation + instance norm.'

Dataset                   Run ID                    Mean Val Dice    Std Val Dice
-------------------------------------------------------------------------------------
AbdomenUS                 2026_07_13_01_47_49_1740137 0.5653           0.0606
AbdomenUS                 2026_07_13_21_08_43_1742118 0.5656           0.1020
                          Δ dice                    +0.0003

Acdc                      2026_07_13_01_47_40_1740133 0.8567           0.0073
Acdc                      2026_07_13_21_08_14_1742114 0.8577           0.0180
                          Δ dice                    +0.0010

BAGLS                     (no runs found)

Bbbc010                   2026_07_13_01_47_37_1740138 0.8219           0.0276
Bbbc010                   2026_07_13_21_08_43_1742119 0.7895           0.0356
                          Δ dice                    -0.0324

BkaiIgh                   2026_07_13_01_47_39_1740139 0.7295           

In [8]:
path = "experiments/experiments_large.csv"

with open(path) as file:
    lines = file.readlines()

header = lines[0].split(',')
expected_field_count = len(header)

print(f"Expected {expected_field_count} fields per row\n")

problematic_lines = []

for index, line in enumerate(lines[1:], start=2):
    field_count = line.count(',') + 1
    if field_count != expected_field_count:
        problematic_lines.append((index, field_count, line[:150]))

print(f"Found {len(problematic_lines)} problematic lines:\n")
for line_number, field_count, preview in problematic_lines:
    print(f"Line {line_number}: {field_count} fields (expected {expected_field_count})")
    print(f"  Preview: {repr(preview)}\n")

Expected 44 fields per row

Found 1141 problematic lines:

Line 2: 47 fields (expected 44)
  Preview: '2026_06_12_16_29_37,2026-06-12,EMSegmentation,45481,0.1667,0.0317,Depth-wise separable layers does not lose much in Dice.,emseg + no norm + ds conv 4 '

Line 3: 47 fields (expected 44)
  Preview: '2026_06_12_17_19_41,2026-06-12,EMSegmentation,46697,0.8973,0.012,Depth-wise separable layers does not lose much in Dice.,emseg + batch norm + ds conv '

Line 4: 46 fields (expected 44)
  Preview: '2026_06_13_12_25_23,2026-06-13,EMSegmentation,74321,0.8602,0.0358,Depth-wise separable layers does not lose much in Dice.,emseg + no norm + standard c'

Line 5: 46 fields (expected 44)
  Preview: '2026_06_13_12_46_19,2026-06-13,EMSegmentation,74897,0.9153,0.005,Depth-wise separable layers does not lose much in Dice.,emseg + batch norm + standard'

Line 6: 46 fields (expected 44)
  Preview: '2026_06_13_12_59_21,2026-06-13,EMSegmentation,12617,0.1756,0.0,Depth-wise separable layers does not lose much

In [9]:
import csv

path = "experiments/experiments_large.csv"

with open(path) as file:
    reader = csv.reader(file)
    header = next(reader)
    expected_field_count = len(header)
    
    print(f"Expected {expected_field_count} fields per row\n")
    
    problematic_lines = []
    
    for line_number, row in enumerate(reader, start=2):
        if len(row) != expected_field_count:
            problematic_lines.append((line_number, len(row), row[:3]))
    
    print(f"Found {len(problematic_lines)} problematic lines:\n")
    for line_number, field_count, preview in problematic_lines:
        print(f"Line {line_number}: {field_count} fields (expected {expected_field_count})")
        print(f"  Preview: {preview}\n")

Expected 44 fields per row

Found 0 problematic lines:

